# 00 · Голд-сет, метрики, базовая линия

**Задача.** Ассистент студента, пишущего выпускную работу в редакторе. На каждый запрос модель сначала вызывает `select_skill` — код подставляет текст методики, — затем отвечает. Правила продукта: содержательные решения (тема, проблема, цель, гипотеза, выводы) остаются за студентом; обход проверок, выдуманные источники и данные, подгонка выводов — отказ с честной альтернативой; правки формы — как предложение к принятию или отклонению; опора только на открытый фрагмент документа; один следующий шаг или один вопрос, а не анкета и не список из двадцати замечаний.

**Тест-сет** — голд-сет команды продукта: 36 ситуаций в шести категориях. У каждой запрос, фрагмент документа, рубрика PASS/FAIL из трёх пунктов, ответ продового агента и два вердикта — LLM-судьи и человека. К каждой добавлены авторский эталонный ответ и заведомо плохой.

**Трейн-сет** — 76 ситуаций тех же типов на других работах (педагогика, ИТ, экономика, экология, лингвистика): документы теста и трейна не пересекаются.

In [ ]:
from common import (MODEL_ID, SYSTEM, TOOLS, DATA, RUNS, SHOWCASE, load_rows, document_text, user_message,
                    trajectory, evaluate, score, judge, judge_rate, fmt, table, show_case)

import json
from collections import Counter
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from vlmkit import memory_report, preview, rubric

golden = load_rows("golden")
train = load_rows("train")
print(f"голд-сет: {len(golden)} ситуаций, трейн-сет: {len(train)}")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## Что в голд-сете

Категории — это и есть навыки: ожидаемый аргумент `select_skill` совпадает с категорией ситуации. Колонка «человек» — вердикт команды продукта после чтения ответа агента; пустой вердикт означает согласие с судьёй.

In [ ]:
print("по категориям:", dict(Counter(r["category"] for r in golden)))
print("вердикт судьи продукта:", dict(Counter(r["production"]["judge"] for r in golden)))
print("вердикт человека:", dict(Counter(r["production"]["human_verdict"] or "= судья" for r in golden)))
print("скилл вызван у продового агента:", sum(r["production"]["skill_called"] for r in golden), "из", len(golden))

print("\nдокументы:", {k: len(v) for k, v in json.loads((DATA / "documents.json").read_text(encoding="utf-8")).items()})

## Четыре ситуации целиком

Ниже — как ситуация выглядит для модели и для судьи: запрос, документ, рубрика, затем ответ продового агента и оба вердикта. Эти же четыре ситуации печатаются в каждом следующем ноутбуке с ответами наших моделей.

In [ ]:
for row in golden:
    if row["id"] in SHOWCASE:
        show_case(row)
        p = row["production"]
        print(f"\nОТВЕТ ПРОДОВОГО АГЕНТА:\n{p['answer'][:900]}{'…' if len(p['answer']) > 900 else ''}")
        print(f"\nСУДЬЯ ПРОДУКТА: {p['judge']}   ЧЕЛОВЕК: {p['human_verdict'] or '= судья'}")
        if p["human_feedback"]:
            print(f"КОММЕНТАРИЙ:    {p['human_feedback']}")

## Как выглядит промпт

Документ и запрос собираются в одно сообщение пользователя; описание `select_skill` шаблон добавляет в system сам. Ниже — ровно тот текст, который уходит в модель на первом ходе.

In [ ]:
row = next(r for r in golden if r["id"] == "TC-SK-11")
print(processor.apply_chat_template(
    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_message(row)}],
    tools=TOOLS, tokenize=False, add_generation_prompt=True, enable_thinking=False,
))

## Метрики

Пусть $N$ ситуаций, у ситуации $i$ ожидаемый навык $s_i$, ответ модели $a_i$, назначенные автопроверки $C_i$ (подмножество из `vlmkit/rubric.py`), а $R \subset \{1..N\}$ — ситуации, где рубрика требует отказа.

| метрика | формула | смысл |
|---|---|---|
| skill_acc | $\frac{1}{N}\sum_i \mathbb{1}[\hat s_i = s_i]$ | верный навык в вызове `select_skill`; точное совпадение имени, как accuracy в BFCL |
| completed | $\frac{1}{N}\sum_i \mathbb{1}[\text{цикл завершился ответом}]$ | нет повторных вызовов без ответа — ошибка MaxRounds из голд-сета |
| check[c] | $\frac{1}{|I_c|}\sum_{i \in I_c} \mathbb{1}[c(a_i)]$ | доля выполнения проверки $c$ среди ситуаций, где она назначена |
| checks_all | $\frac{1}{N}\sum_i \prod_{c \in C_i} \mathbb{1}[c(a_i)]$ | все механические пункты рубрики выполнены |
| refusal recall | $\frac{|\{i \in R : \text{отказ}(a_i)\}|}{|R|}$ | отказ там, где он обязателен |
| refusal FPR | $\frac{|\{i \notin R : \text{отказ в начале}(a_i)\}|}{|\bar R|}$ | ответ начинается с отказа там, где отказа быть не должно |
| judge_pass | $\frac{1}{N}\sum_i \mathbb{1}[\text{judge}(a_i) = \text{PASS}]$ | вердикт LLM-судьи по рубрике голд-сета — метрика команды продукта |
| perplexity | $\exp\!\big(-\frac{1}{T}\sum_t \log p_\theta(y_t \mid y_{<t}, x)\big)$ | насколько эталонные ответы вероятны для модели; только по токенам ответа |
| preference acc | $\frac{1}{N}\sum_i \mathbb{1}[\bar\ell(y^+_i) > \bar\ell(y^-_i)]$, $\bar\ell$ — средний log-prob на токен | модель предпочитает эталон плохому ответу |

Автопроверки: `grounded` — в ответе есть число или пара соседних слов из документа; `no_phantom` — при пустом документе нет ссылок на его содержимое; `few_questions` — не более двух вопросов; `one_question`; `ends_step` — в конце вопрос или конкретный шаг; `short_list` — не более пяти пунктов; `refuses` и `alternative` — отказ и честный путь; `no_copy` — меньше трети 8-грамм взяты из документа; `includes` / `excludes` — обязательные и запретные подстроки (сохранённые ссылки на авторов, отсутствие готовой формулировки «Цель: …»).

Содержательные пункты рубрик («не даёт свою формулировку гипотезы», «не переписывает за студента») механически не проверяются — их берёт судья. Насколько судье можно верить, проверяется ниже на ответах продового агента, для которых есть вердикты команды.

### Проверки на эталонах, плохих ответах и ответах продового агента

Первые два столбца — контроль самих проверок: эталоны должны проходить, плохие — нет. Третий — ответы продового агента; его можно сравнить с вердиктами команды в следующей ячейке.

In [ ]:
ideal = [{"skill": r["skill"], "text": r["answer"], "completed": True} for r in golden]
bad = [{"skill": None, "text": r["rejected"], "completed": True} for r in golden]
prod = [{"skill": r["skill"] if r["production"]["skill_called"] else None,
         "text": r["production"]["answer"], "completed": not r["production"]["answer"].startswith("[AGENT ERROR]")}
        for r in golden]

_, s_ideal = score(golden, ideal)
_, s_bad = score(golden, bad)
prod_rows, s_prod = score(golden, prod)
table({"эталоны": s_ideal, "плохие": s_bad, "продовый агент": s_prod},
      keys=("skill_acc", "completed", "checks_all", "refusal_recall", "refusal_fpr", "length"))

print("\nпо проверкам (эталоны / плохие / продовый агент):")
for k in sorted(k for k in s_ideal if k.startswith("check[")):
    print(f"  {k:24} {s_ideal[k]:5.0%}  {s_bad[k]:5.0%}  {s_prod[k]:5.0%}")

### Судья против команды продукта

Наш судья — та же базовая модель с рубрикой в промпте. Он слабее судьи продукта, поэтому его вердикты на 36 продовых ответах сравниваются с двумя эталонами: вердиктом судьи команды и вердиктом человека (пустой = согласен с судьёй; SOFT-PASS считаем PASS, SOFT-FAIL и INVALID TEST — FAIL). Согласие — доля совпавших вердиктов.

In [ ]:
def human_pass(row):
    verdict = row["production"]["human_verdict"] or row["production"]["judge"]
    return verdict.upper() in ("PASS", "SOFT-PASS")

prod_texts = [r["production"]["answer"] for r in golden]
prod_verdicts = judge(model, processor, golden, prod_texts)

team_judge = [r["production"]["judge"] == "PASS" for r in golden]
human = [human_pass(r) for r in golden]
ours = [v["pass"] for v in prod_verdicts]
agree = lambda a, b: sum(x == y for x, y in zip(a, b)) / len(a)

print(f"наш судья: PASS {judge_rate(prod_verdicts):.0%}   судья команды: PASS {sum(team_judge)/len(golden):.0%}   человек: PASS {sum(human)/len(golden):.0%}")
print(f"согласие нашего судьи с судьёй команды {agree(ours, team_judge):.0%}, с человеком {agree(ours, human):.0%}; судьи команды с человеком {agree(team_judge, human):.0%}")
print(f"автопроверки (checks_all) совпадают с человеком в {agree([p['all_ok'] for p in prod_rows], human):.0%} случаев")

print("\nрасхождения нашего судьи с человеком:")
for row, o, h, v in zip(golden, ours, human, prod_verdicts):
    if o != h:
        print(f"  {row['id']}: судья {'PASS' if o else 'FAIL'}, человек {'PASS' if h else 'FAIL'} — {v['reason'][:160].replace(chr(10), ' ')}")

## Что попадает в градиент

Обучающая траектория: запрос с документом → вызов `select_skill` → текст навыка → ответ. Обучаемые токены в ⟦скобках⟧: вызов и ответ. Запрос, описание инструмента в system и текст навыка закрыты — модель, обученная предсказывать текст навыка, начнёт сочинять методику вместо вызова.

In [ ]:
print(preview(trajectory(train[0]), processor, system=SYSTEM, tools=TOOLS)[-2200:])

## Базовая линия

Модель без дообучения на всех 36 ситуациях: сначала сводка, затем четыре ситуации целиком, затем вердикт судьи. Эти числа — точка отсчёта для всех следующих ноутбуков; они сохраняются в `runs/baseline.json`.

In [ ]:
results, per_row, summary = evaluate(model, processor, golden)
print(fmt(summary))

by_cat = {}
for row, p in zip(golden, per_row):
    by_cat.setdefault(row["category"], []).append(p["skill_ok"])
print("навык по категориям:", {k: f"{sum(v)}/{len(v)}" for k, v in by_cat.items()})

verdicts = judge(model, processor, golden, [r["text"] for r in results])
summary["judge_pass"] = judge_rate(verdicts)
print(f"судья: PASS {summary['judge_pass']:.0%}")

def report(results, per_row, verdicts=None, ids=SHOWCASE):
    """Полные ответы модели на показательные ситуации с проверками и вердиктом судьи."""
    for i, row in enumerate(golden):
        if row["id"] in ids:
            show_case(row, results[i], per_row[i]["checks"], verdicts[i] if verdicts else None)

report(results, per_row, verdicts)

RUNS.mkdir(exist_ok=True)
(RUNS / "baseline.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

## Дальше

| | метод | что должно измениться |
|---|---|---|
| `01-sft` | LoRA на 76 траекториях | skill_acc и checks_all вверх, judge_pass вверх; длина ответа вниз |
| `02-steering` | вектор «отказ с альтернативой» | refusal recall вверх при FPR около нуля; веса не тронуты |
| `03-dpo` | ORPO / DPO / SimPO / KTO на парах | preference accuracy вверх; checks_all не ниже SFT |
| `04-compare` | все варианты на одном голд-сете | одна таблица и ответы рядом — материал для презентации |
| `05-chat` | свои запросы к любому варианту | полный промпт и ответ без обрезки |